# Particle Filter Use Cases — TFiltersPy

This notebook demonstrates the **`ParticleFilter`** API from TFiltersPy on three real-world
use cases: image denoising (computer vision), multivariate EEG time-series filtering,
and NLP topic tracking.

**Particle filters** approximate the posterior distribution with a set of weighted
samples (particles). They handle **nonlinear, non-Gaussian** models where Kalman
variants struggle, at the cost of higher computation (each timestep propagates
`n_particles` samples). For the linear-Gaussian examples below a standard Kalman
filter would be faster and equally accurate — the particle filter is used here to
illustrate the API and to show it converges to similar results.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pandas as pd

from tfilterspy import ParticleFilter

np.random.seed(42)
print("Imports OK")

---

## 1. Image Denoising (Computer Vision)

We treat each 8x8 digit image (64 pixels) as a state vector observed through
additive Gaussian noise. The particle filter estimates the clean image from
the noisy observations.

**Model:**
$$
x_{k+1} = I_{64}\, x_k + w_k, \quad w_k \sim \mathcal{N}(0,\, 0.01\,I)
$$
$$
z_k = I_{64}\, x_k + v_k, \quad v_k \sim \mathcal{N}(0,\, 0.1\,I)
$$

Because both `f` and `h` are identity matrices, the filter auto-vectorizes
particle propagation (matrix path), which is much faster than the callable path
for high-dimensional states.

> **Note:** For a purely linear-Gaussian model like this, the Kalman filter is
> analytically optimal *and* runs in O(n^3) per step rather than O(n^2 * N_particles).
> The particle filter is shown here as an API demo and to verify it converges to
> comparable results.

In [ ]:
# Load digits (20 samples for speed — 64-dimensional state)
digits = load_digits()
x = digits.data[:20]    # (20, 64)
y = digits.target[:20]  # (20,)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# Add Gaussian noise
noise_level = 0.88
noisy_train = x_train + np.random.normal(0, noise_level, x_train.shape)
noisy_test  = x_test  + np.random.normal(0, noise_level, x_test.shape)

print(f"Train: {x_train.shape}, Test: {x_test.shape}")
print(f"Noise level: {noise_level}")

In [ ]:
# Build ParticleFilter with matrix-based f and h (auto-vectorized)
n_features = 64
F = np.eye(n_features)           # Static transition
H = np.eye(n_features)           # Direct observation
Q = np.eye(n_features) * 0.01    # Process noise covariance
R = np.eye(n_features) * 0.1     # Observation noise covariance
x0 = x_train[0]                  # First clean image as initial state

pf = ParticleFilter(
    f=F, h=H, Q=Q, R=R,
    x0=x0, n_particles=500
)

# Run the particle filter on noisy training images
pf.fit(noisy_train)
denoised_train = pf.predict()  # Filtered state estimates

print(f"Denoised shape: {denoised_train.shape}")
print(f"ESS (last step): {pf.effective_sample_sizes_[-1]:.1f} / {pf.n_particles}")

In [ ]:
# Visualize: Original / Noisy / Denoised for 4 training images
n_display = 4
fig, axes = plt.subplots(3, n_display, figsize=(12, 7))

row_labels = ["Original", "Noisy", "Denoised (PF)"]
for i in range(n_display):
    axes[0, i].imshow(x_train[i].reshape(8, 8), cmap="gray")
    axes[1, i].imshow(noisy_train[i].reshape(8, 8), cmap="gray")
    axes[2, i].imshow(denoised_train[i].reshape(8, 8), cmap="gray")
    for row in range(3):
        axes[row, i].axis("off")

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=12, rotation=0, labelpad=80, va="center")

fig.suptitle("Particle Filter Image Denoising (digits, 64D)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Compute MSE
mse_noisy    = np.mean((x_train - noisy_train) ** 2)
mse_denoised = np.mean((x_train - denoised_train) ** 2)
print(f"MSE (noisy vs clean):    {mse_noisy:.4f}")
print(f"MSE (denoised vs clean): {mse_denoised:.4f}")
print(f"Reduction:               {(1 - mse_denoised / mse_noisy) * 100:.1f}%")

# Score (negative MSE, sklearn convention — higher is better)
score = pf.score(x_train)
print(f"\npf.score(true): {score:.4f}")

In [ ]:
# Classify denoised images with Logistic Regression
# Re-run PF on test set
pf_test = ParticleFilter(
    f=F, h=H, Q=Q, R=R,
    x0=x_test[0], n_particles=500
)
pf_test.fit(noisy_test)
denoised_test = pf_test.predict()

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(denoised_train, y_train)

acc_denoised = clf.score(denoised_test, y_test)
acc_noisy    = LogisticRegression(max_iter=1000, random_state=42).fit(
    noisy_train, y_train
).score(noisy_test, y_test)

print(f"Classification accuracy (noisy):    {acc_noisy:.4f}")
print(f"Classification accuracy (denoised): {acc_denoised:.4f}")

---

## 2. Multivariate EEG Time Series

The [EEG Eye State](https://www.openml.org/d/1471) dataset contains 14 EEG
channels. We add synthetic noise and use the particle filter to recover the
original signal.

**Model:**
$$
x_{k+1} = 0.99\,I_{14}\, x_k + w_k
$$
$$
z_k = I_{14}\, x_k + v_k
$$

The 0.99 factor introduces slight decay, which helps the filter avoid
divergence on non-stationary EEG signals.

> **Speed note:** Particle filters scale as O(T * N * d) where T is timesteps,
> N is particle count, and d is state dimension. We limit to 200 rows and
> 500 particles to keep runtime reasonable.

In [ ]:
# Load EEG data (first 200 rows for PF speed)
eeg_data = fetch_openml(
    name='eeg-eye-state', version=1,
    as_frame=False, parser='liac-arff'
)
X_eeg = eeg_data.data[:200].astype(np.float64)  # (200, 14)
print(f"EEG data shape: {X_eeg.shape}")

# Add synthetic noise
noise_level_eeg = 0.5
noisy_eeg = X_eeg + np.random.normal(0, noise_level_eeg, X_eeg.shape)

In [ ]:
# Build ParticleFilter for 14-channel EEG
n_ch = 14
F_eeg = np.eye(n_ch) * 0.99   # Slight decay
H_eeg = np.eye(n_ch)          # Direct observation
Q_eeg = np.eye(n_ch) * 0.01
R_eeg = np.eye(n_ch) * 0.1
x0_eeg = X_eeg[0]             # First clean sample as initial state

pf_eeg = ParticleFilter(
    f=F_eeg, h=H_eeg, Q=Q_eeg, R=R_eeg,
    x0=x0_eeg, n_particles=500
)

pf_eeg.fit(noisy_eeg)
denoised_eeg = pf_eeg.predict()

mse_eeg = np.mean((X_eeg - denoised_eeg) ** 2)
print(f"EEG denoising MSE: {mse_eeg:.4f}")

In [ ]:
# Plot: Raw vs Noisy vs Denoised for 4 channels
channels_to_plot = [0, 3, 7, 13]
fig, axes = plt.subplots(len(channels_to_plot), 1, figsize=(14, 10), sharex=True)

for idx, ch in enumerate(channels_to_plot):
    axes[idx].plot(X_eeg[:, ch], label="Original", alpha=0.6)
    axes[idx].plot(noisy_eeg[:, ch], label="Noisy", alpha=0.4)
    axes[idx].plot(denoised_eeg[:, ch], label="Denoised (PF)", linestyle="--", linewidth=2)
    axes[idx].set_ylabel(f"Ch {ch + 1}")
    axes[idx].legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Timestep")
fig.suptitle("EEG Denoising — Particle Filter (200 steps, 500 particles)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Effective Sample Size (ESS) over time
plt.figure(figsize=(12, 3))
plt.plot(pf_eeg.effective_sample_sizes_, color="teal")
plt.axhline(y=pf_eeg.n_particles * pf_eeg.resample_threshold,
            color="red", linestyle="--", label="Resample threshold")
plt.xlabel("Timestep")
plt.ylabel("ESS")
plt.title("Effective Sample Size — EEG Particle Filter")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean ESS: {np.mean(pf_eeg.effective_sample_sizes_):.1f}")
print(f"Min  ESS: {np.min(pf_eeg.effective_sample_sizes_):.1f}")

---

## 3. NLP Topic Tracking (Disaster Tweets)

We extract 5 LDA topics from disaster tweets and treat the sequence of topic
distributions as a time series. The particle filter smooths the noisy topic
probabilities, revealing stable trends.

**Model:**
$$
x_{k+1} = 0.95\,I_5\, x_k + w_k
$$
$$
z_k = I_5\, x_k + v_k
$$

The 0.95 decay encourages the filter to slowly forget old topic distributions,
adapting to topic drift in the tweet stream.

> **Why PF over KF here?** In practice, topic distributions are constrained
> to the simplex (probabilities sum to 1) and are not truly Gaussian. A particle
> filter can, in principle, enforce simplex constraints via the proposal
> distribution or resampling — though this basic example uses the standard
> Gaussian model for simplicity.

In [ ]:
# Load disaster tweets
data_path = '../data/train_nlp.csv'
df = pd.read_csv(data_path)
tweets = df['text'].values[:500]  # 500 tweets (small for PF speed)
print(f"Number of tweets: {len(tweets)}")

# Extract 5 LDA topics
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X_bow = vectorizer.fit_transform(tweets)

n_topics = 5
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
topic_dist = lda.fit_transform(X_bow)  # (500, 5)
print(f"Topic distribution shape: {topic_dist.shape}")

In [ ]:
# Build ParticleFilter for topic tracking
F_nlp = np.eye(n_topics) * 0.95   # Slight decay for topic drift
H_nlp = np.eye(n_topics)          # Direct observation
Q_nlp = np.eye(n_topics) * 0.01
R_nlp = np.eye(n_topics) * 0.1
x0_nlp = topic_dist[0]            # First tweet's topic distribution

pf_nlp = ParticleFilter(
    f=F_nlp, h=H_nlp, Q=Q_nlp, R=R_nlp,
    x0=x0_nlp, n_particles=500
)

pf_nlp.fit(topic_dist)
filtered_topics = pf_nlp.predict()

print(f"Filtered topics shape: {filtered_topics.shape}")

In [ ]:
# Plot raw vs filtered topic distributions
fig, axes = plt.subplots(n_topics, 1, figsize=(14, 10), sharex=True)

for i in range(n_topics):
    axes[i].plot(topic_dist[:, i], label=f"Raw Topic {i + 1}", alpha=0.5)
    axes[i].plot(filtered_topics[:, i], label=f"Filtered Topic {i + 1}",
                 linestyle="--", linewidth=2)
    axes[i].set_ylabel(f"Topic {i + 1}")
    axes[i].legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Tweet index (temporal order)")
fig.suptitle("Topic Tracking — Raw LDA vs Particle-Filtered", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ESS plot for NLP topic tracking
plt.figure(figsize=(12, 3))
plt.plot(pf_nlp.effective_sample_sizes_, color="darkorange")
plt.axhline(y=pf_nlp.n_particles * pf_nlp.resample_threshold,
            color="red", linestyle="--", label="Resample threshold")
plt.xlabel("Tweet index")
plt.ylabel("ESS")
plt.title("Effective Sample Size — NLP Topic Particle Filter")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean ESS: {np.mean(pf_nlp.effective_sample_sizes_):.1f}")
print(f"Min  ESS: {np.min(pf_nlp.effective_sample_sizes_):.1f}")

In [ ]:
# Top words per topic
feature_names = vectorizer.get_feature_names_out()
print("Top words per LDA topic:")
print("-" * 50)
for i, topic in enumerate(lda.components_):
    top_words = [feature_names[j] for j in topic.argsort()[-8:][::-1]]
    print(f"  Topic {i + 1}: {', '.join(top_words)}")

---

## Summary

| Use Case | State dim | Particles | Key observation |
|---|---|---|---|
| Image denoising | 64 | 500 | PF recovers clean digits; MSE drops significantly vs noisy input |
| EEG time series | 14 | 500 | PF smooths noisy channels; ESS shows when resampling triggers |
| NLP topic tracking | 5 | 500 | PF reveals stable topic trends hidden by LDA sampling noise |

**When to prefer a Particle Filter over a Kalman Filter:**
- Non-Gaussian noise (e.g., outliers, heavy tails)
- Nonlinear state transitions or observation models
- Multimodal posteriors
- Constrained state spaces (e.g., simplex for topic distributions)

**When to prefer a Kalman Filter:**
- Linear-Gaussian models (KF is optimal and much faster)
- Very high-dimensional states (PF suffers from the curse of dimensionality)
- Real-time applications where O(n^3) per step is acceptable